### Garbage Sorting Gameplay

This is garbage sorting 

### Import Header Files


In [1]:
#!/usr/bin/env python
# coding: utf-8
# Install the missing module
%pip install Arm_Lib
%pip install threading
%pip install ipywidget

Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement threading (from versions: none)
ERROR: No matching distribution found for threading
Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement ipywidget (from versions: none)
ERROR: No matching distribution found for ipywidget
Note: you may need to restart the kernel to use updated packages.


In [ ]:

# Import the necessary libraries
import Arm_Lib
import threading
import time
import random
# Create a class to handle the arm movements
class ArmMovement:
    def __init__(self, arm):
        self.arm = arm
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self.move_arm)
        self.thread.start()

    def move_arm(self):
        while not self.stop_event.is_set():
            # Generate random angles for the arm joints
            angles = [random.uniform(-180, 180) for _ in range(6)]
            # Move the arm to the new angles
            self.arm.move_to_angles(angles)
            # Sleep for a short duration before moving again
            time.sleep(1)

    def stop(self):
        self.stop_event.set()
        self.thread.join()
# Initialize the arm
arm = Arm_Lib.Arm()
# Create an instance of the ArmMovement class
arm_movement = ArmMovement(arm)
# Run the arm movement for a while
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    # Stop the arm movement when interrupted
    arm_movement.stop()
    print("Arm movement stopped.")
# Clean up
arm.stop()
arm.close()
# This code is a simple example of how to control a robotic arm using threading and random movements.
# It creates a class that handles the arm movements in a separate thread, allowing for continuous movement until interrupted.
# The arm is moved to random angles in a loop, and the movement can be stopped gracefully using a stop event.





In [ ]:

import Arm_Lib # type: ignore
import cv2 as cv
import threading
from time import sleep
import ipywidgets as widgets
from IPython.display import display
from single_garbage_identify import single_garbage_identify


### Create Instance and Initialize Parameters



In [ ]:
# Create single_garbage_identify instance
# 
single_garbage = single_garbage_identify()
# Load the model
# single_garbage.load_model("model/garbage_model.h5")
# Load the model
model = "General"

### Initialize Robotic Arm Position

In [ ]:
import Arm_Lib
arm = Arm_Lib.Arm_Device()
joints_0 = [90, 90, 20, 15, 90, 30]
arm.Arm_serial_servo_write6_array(joints_0, 1000)


### Creating widgets


In [ ]:
# Create the widgets
button_layout      = widgets.Layout(width='320px', height='60px', align_self='center')
output = widgets.Output()
# Exit button
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
controls_box = widgets.VBox([imgbox, exit_button], layout=widgets.Layout(align_self='center'))

### Exit Mode Widget

In [ ]:
def exit_button_Callback(value):
    global model
    model = 'Exit'
    with output: print(model)
exit_button.on_click(exit_button_Callback)



### Main Program

In [ ]:
def camera():
    # Create a VideoCapture object to read from the camera
    capture = cv.VideoCapture(0)
    # Loop to read frames from the camera
    while capture.isOpened():
        try:
            # Read the image from the camera
            _, img = capture.read()
            # Standardize the image size
            img = cv.resize(img, (640, 480))
            img = single_garbage.single_garbage_run(img)
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except KeyboardInterrupt:capture.release()

### Launch

In [ ]:
display(controls_box,output)
threading.Thread(target=camera, ).start()